# 01 — Dự đoán giá nhà (Supervised ML)

## Mục tiêu
- Hiểu pipeline: dữ liệu → feature/label → train/test → model → đánh giá → dự đoán mới.
- Dùng Scikit-learn trên bài toán hồi quy (regression).

## Kiến thức cần nhớ
- **Supervised learning**: có nhãn (ở đây là `gia_trieu_vnd`).
- **Feature** = đầu vào (diện tích, số phòng…); **label** = giá trị cần đoán.
- **Train/test split**: giữ một phần dữ liệu để kiểm tra model có bị nhớ máy không.
- **Overfitting**: điểm train rất tốt nhưng test kém — model “học thuộc” thay vì khái quát.

Công cụ: Pandas, Scikit-learn (Random Forest).


## 1. Load và xem dữ liệu


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data/nha_gia.csv")
df = pd.read_csv(DATA)
print(df.shape)
df.head()


In [ ]:
# EDA ngan
print(df["gia_trieu_vnd"].describe())
print("\nTuong quan so voi gia:")
print(df.corr(numeric_only=True)["gia_trieu_vnd"].sort_values(ascending=False))


## 2. Chuẩn bị feature / label

Cột `quan` là categorical → one-hot encode. Các cột số giữ nguyên.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

feature_cols = ["dien_tich_m2", "so_phong_ngu", "so_phong_tam", "tuoi_nha_nam", "so_tang", "quan"]
label_col = "gia_trieu_vnd"

X = df[feature_cols]
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)


## 3. Pipeline: encode + Random Forest


In [ ]:
numeric_features = ["dien_tich_m2", "so_phong_ngu", "so_phong_tam", "tuoi_nha_nam", "so_tang"]
categorical_features = ["quan"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42)),
    ]
)

model.fit(X_train, y_train)
print("Da train xong.")


## 4. Đánh giá MAE / RMSE


In [ ]:
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
print(f"MAE  = {mae:.2f} trieu VND  (sai so tuyet doi trung binh)")
print(f"RMSE = {rmse:.2f} trieu VND")

# So sanh nhanh 5 mau
so_sanh = X_test.copy()
so_sanh["gia_that"] = y_test.values
so_sanh["gia_du_doan"] = pred.round(1)
so_sanh.head()


## 5. Dự đoán căn nhà mới

Nhập số phòng ngủ, diện tích, quận… giống tính năng định giá.


In [ ]:
nha_moi = pd.DataFrame(
    [
        {
            "dien_tich_m2": 72.0,
            "so_phong_ngu": 2,
            "so_phong_tam": 2,
            "tuoi_nha_nam": 5,
            "so_tang": 2,
            "quan": "Quan 7",
        }
    ]
)
gia_du_doan = model.predict(nha_moi)[0]
print(f"Gia du doan: {gia_du_doan:.1f} trieu VND")


## Giải thích cho bạn (đọc to)

1. Model **không** nhớ từng căn nhà; nó học quy luật từ feature → giá.
2. MAE ~ "trung bình mỗi lần đoán lệch bao nhiêu triệu".
3. Nếu train rất tốt mà test kém → nghi **overfitting** (thử model đơn giản hơn hoặc thêm data).


## Bài tập nhỏ

1. Đổi `RandomForestRegressor` sang `LinearRegression` (import từ `sklearn.linear_model`) và so sánh MAE.
2. Thử căn nhà ở `Go Vap`, 45m2, 1 phòng ngủ — giá dự đoán thay đổi thế nào?
3. Vẽ scatter `y_test` vs `pred` bằng matplotlib (tuỳ chọn).


In [ ]:
# TODO: bai tap
# from sklearn.linear_model import LinearRegression
# ...


## Checklist — đã hiểu khi…
- [ ] Nói được feature / label / train-test trong bài này là gì
- [ ] Chạy được pipeline và đọc MAE
- [ ] Dự đoán được 1 căn nhà mới bằng `model.predict`
- [ ] Giải thích overfitting bằng 1 câu
